In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :exponential

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model] Fitting chain 1 (tau=34)
[ Info: [exponential] iter 1000/1000000 elapsed=3.9s, rate=0.159, mean=[1.649, 0.00123, 1.531, 0.128], std=[0.2469, 0.000268, 0.0471, 0.0654] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=6.9s, rate=0.126, mean=[1.878, 0.00107, 1.469, 0.107], std=[0.2754, 0.000248, 0.0690, 0.0501] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=9.1s, rate=0.111, mean=[1.928, 0.00103, 1.446, 0.099], std=[0.2379, 0.000216, 0.0647, 0.0422] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=11.3s, rate=0.104, mean=[1.970, 0.00100, 1.438, 0.096], std=[0.2166, 0.000199, 0.0578, 0.0369] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=13.5s, rate=0.103, mean=[2.008, 0.00098, 1.452, 0.094], std=[0.2056, 0.000186, 0.0594, 0.0333] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=15.7s, rate=0.102, mean=[2.030, 0.00098, 1.484, 0.092], std=[0.1945, 0.000175, 0.0943, 0.0308] [ADAPT]
[ Info: [exponential] iter 7000/1000000 elap